# 04 — Model Evaluation & Analysis

**Student Academic Risk — Early Intervention System**

This notebook performs deep evaluation of the best-performing model:
- Detailed classification report
- ROC and Precision-Recall curves
- Decision boundary analysis
- Error analysis
- Prediction examples
- Final model summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score
)
from sklearn.model_selection import learning_curve, StratifiedKFold

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

## 1. Load Model and Data

In [ ]:
models_dir = os.path.join('..', 'data_science', 'models')
processed_dir = os.path.join('..', 'data_science', 'data', 'processed')

# Load model and artifacts
best_model = joblib.load(os.path.join(models_dir, 'best_model.joblib'))
scaler = joblib.load(os.path.join(models_dir, 'scaler.joblib'))
le = joblib.load(os.path.join(models_dir, 'label_encoder.joblib'))
feature_cols = joblib.load(os.path.join(models_dir, 'feature_columns.joblib'))
metadata = joblib.load(os.path.join(models_dir, 'model_metadata.joblib'))

# Load data
X_train = np.load(os.path.join(processed_dir, 'X_train.npy'))
X_test = np.load(os.path.join(processed_dir, 'X_test.npy'))
y_train = np.load(os.path.join(processed_dir, 'y_train.npy'))
y_test = np.load(os.path.join(processed_dir, 'y_test.npy'))

# Load full processed data for analysis
df = pd.read_csv(os.path.join(processed_dir, 'processed_data.csv'))

print(f'Model: {metadata["model_name"]}')
print(f'Classes: {metadata["classes"]}')
print(f'Features: {len(feature_cols)}')
print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')

## 2. Detailed Classification Report

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print(f'\n{"="*50}')
print(f'  {metadata["model_name"]} — Final Test Evaluation')
print(f'{"="*50}')
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
           xticklabels=le.classes_, yticklabels=le.classes_,
           linewidths=2, linecolor='white')
ax.set_title(f'Confusion Matrix — {metadata["model_name"]}', fontweight='bold', fontsize=14)
ax.set_ylabel('Actual', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.savefig('../data_science/data/11_final_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba[:, 1])
roc_auc_val = auc(fpr, tpr)

axes[0].plot(fpr, tpr, color='#6366F1', lw=2, label=f'ROC (AUC = {roc_auc_val:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Baseline')
axes[0].fill_between(fpr, tpr, alpha=0.15, color='#6366F1')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold', fontsize=14)
axes[0].legend(loc='lower right')

# Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba[:, 1])
avg_prec = average_precision_score(y_test, y_proba[:, 1])

axes[1].plot(recall_vals, precision_vals, color='#EC4899', lw=2, label=f'PR (AP = {avg_prec:.4f})')
axes[1].fill_between(recall_vals, precision_vals, alpha=0.15, color='#EC4899')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold', fontsize=14)
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.savefig('../data_science/data/12_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Learning Curve

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train, y_train, cv=cv,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='f1_weighted', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#6366F1')
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='#EC4899')
ax.plot(train_sizes, train_mean, 'o-', color='#6366F1', label='Training score')
ax.plot(train_sizes, test_mean, 'o-', color='#EC4899', label='Validation score')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('F1-Score (Weighted)')
ax.set_title(f'Learning Curve — {metadata["model_name"]}', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data_science/data/13_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Error Analysis

In [ ]:
# Which samples were misclassified?
misclassified_idx = np.where(y_pred != y_test)[0]
print(f'Total test samples: {len(y_test)}')
print(f'Correctly classified: {len(y_test) - len(misclassified_idx)}')
print(f'Misclassified: {len(misclassified_idx)}')

if len(misclassified_idx) > 0:
    print(f'\nMisclassified samples:')
    for idx in misclassified_idx[:10]:  # Show first 10
        actual = le.classes_[y_test[idx]]
        predicted = le.classes_[y_pred[idx]]
        prob = y_proba[idx]
        print(f'  Sample {idx}: Actual={actual}, Predicted={predicted}, '
              f'Probabilities=[{prob[0]:.3f}, {prob[1]:.3f}]')
else:
    print('\n🎉 No misclassifications on the test set!')

## 6. Prediction Examples

In [ ]:
# Demonstrate prediction on sample students
print('='*70)
print('  PREDICTION EXAMPLES')
print('='*70)

# Sample some test instances
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), min(5, len(X_test)), replace=False)

for i, idx in enumerate(sample_indices):
    pred_class = le.classes_[y_pred[idx]]
    actual_class = le.classes_[y_test[idx]]
    prob = y_proba[idx]
    
    risk_emoji = '🔴' if pred_class == 'AT_RISK' else '🟢'
    
    print(f'\n--- Student Example {i+1} ---')
    print(f'  Predicted Risk: {risk_emoji} {pred_class}')
    print(f'  Actual Risk:    {actual_class}')
    print(f'  Probability:    AT_RISK={prob[0]:.1%}, LOW={prob[1]:.1%}')
    
    # Recommendations based on prediction
    if pred_class == 'AT_RISK':
        print(f'  📋 Recommended Interventions:')
        print(f'     ✓ Academic counselling')
        print(f'     ✓ Tutoring support')
        print(f'     ✓ Attendance monitoring')
        if prob[0] > 0.8:
            print(f'     ✓ Urgent lecturer consultation')
    else:
        print(f'  📋 Status: Continue monitoring. No urgent intervention needed.')

## 7. Final Model Summary

In [ ]:
print('='*60)
print('  FINAL MODEL SUMMARY')
print('='*60)
print(f'\n  Model:           {metadata["model_name"]}')
print(f'  Target:          {metadata["target"]}')
print(f'  Classes:         {metadata["classes"]}')
print(f'  Features:        {len(feature_cols)}')
print(f'  Training Size:   {X_train.shape[0]} samples')
print(f'  Test Size:       {X_test.shape[0]} samples')
print(f'\n  --- Test Performance ---')
print(f'  Accuracy:        {metadata["accuracy"]:.4f}')
print(f'  Precision:       {metadata["precision"]:.4f}')
print(f'  Recall:          {metadata["recall"]:.4f}')
print(f'  F1-Score:        {metadata["f1_score"]:.4f}')
print(f'  ROC-AUC:         {metadata["roc_auc"]:.4f}')
print(f'\n  --- Model Artifacts ---')
print(f'  📦 best_model.joblib')
print(f'  📦 scaler.joblib')
print(f'  📦 label_encoder.joblib')
print(f'  📦 feature_columns.joblib')
print(f'  📦 model_metadata.joblib')
print(f'\n✅ Model ready for deployment in the web application.')